# Verify CM_model_DOF_clustering_5 mismatch-log fix

Runs `compute_KO_mismatch` and `compute_input_stimulation_mismatch` once each, using the patched module `CM_model_DOF_clustering_5`, and confirms:

1. `KO_mismatch_log.csv` contains exactly **1 header + 30 data rows** (no doubling).
2. `input_stimulation_mismatch_log.csv` contains exactly **1 header + 62 data rows** (no doubling).
3. Match counts equal the expected Pass-2 values: **28/30 KO** and **47/62 stimulation**.

Active parameter set: **Set C** (currently active in `CM_model_DOF_clustering_5.ObjectiveWrapper`). To verify with Set B, swap the scalars in the parameter cell below.

In [1]:
import os
import csv
import numpy as np
import pandas as pd

from CM_model_DOF_clustering_5 import (
    compute_KO_mismatch,
    compute_input_stimulation_mismatch,
    load_ko_and_expectation_data,
    build_weight_vector,
    family_id_vector,
    wlen,
)

In [2]:
# Load the canonical Excel datasets used by calibration.
ko_df, expectations_df = load_ko_and_expectation_data()
print(f'KO datapoints (non-NaN cells in KO_data_condensed2.xlsx): '
      f'{ko_df.drop(columns=["specie"], errors="ignore").notna().sum().sum()}')
valid = expectations_df.isin(["increase", "decrease", "no change"]).sum().sum()
print(f'Stimulation datapoints (valid labels in input_output_relations2.xlsx): {valid}')

KO datapoints (non-NaN cells in KO_data_condensed2.xlsx): 30
Stimulation datapoints (valid labels in input_output_relations2.xlsx): 62


In [3]:
# Active calibrated parameters (Set C scalars + 5 family weights).

# Set B (canonical) - swap in if reverting:
EC50_scalar = 0.587798
n_scalar    = 1.395697
b_scalar    = 0.283201
p_scalar    = 0.653814

calibrated_weights = np.array([1.01915194, 1.09999999, 0.94966909, 1.10, 0.62873844])
w_vec = build_weight_vector(calibrated_weights, family_id_vector)

print('EC50 =', EC50_scalar, ' n =', n_scalar, ' b =', b_scalar, ' p =', p_scalar)
print('weights =', calibrated_weights)
print('w_vec length =', len(w_vec), '(expected', wlen, ')')

EC50 = 0.587798  n = 1.395697  b = 0.283201  p = 0.653814
weights = [1.01915194 1.09999999 0.94966909 1.1        0.62873844]
w_vec length = 163 (expected 163 )


In [4]:
# Run KO mismatch (writes KO_mismatch_log.csv from scratch).
ko_mismatch_pct = compute_KO_mismatch(
    EC50_scalar, n_scalar, b_scalar, w_vec, ko_df, family_id_vector
)
print(f'KO mismatch returned: {ko_mismatch_pct:.2f}%')

KO IL6 -> STAT: expected decrease, model decrease, comparison w/ data match, change=-1.0980
KO IL6 -> migration: expected decrease, model decrease, comparison w/ data match, change=-0.7122
KO IL6 -> proliferation: expected decrease, model decrease, comparison w/ data match, change=-0.5911
KO IL6 -> aSMA: expected decrease, model decrease, comparison w/ data match, change=-0.0404
KO IL6 -> TIMP1: expected decrease, model decrease, comparison w/ data match, change=-0.1658
KO IL6 -> proMMP1: expected decrease, model decrease, comparison w/ data match, change=-0.3144
KO IL6 -> MMP1: expected decrease, model decrease, comparison w/ data match, change=-0.2174
KO TNFa -> TGFB: expected decrease, model decrease, comparison w/ data match, change=-0.3231
KO TNFa -> aSMA: expected decrease, model decrease, comparison w/ data match, change=-0.0404
KO TNFa -> CImRNA: expected decrease, model decrease, comparison w/ data match, change=-0.0856
KO TNFa -> CIIImRNA: expected decrease, model decrease, c

In [5]:
# Run input-stimulation mismatch (writes input_stimulation_mismatch_log.csv from scratch).
stim_mismatch_pct = compute_input_stimulation_mismatch(
    EC50_scalar, n_scalar, b_scalar, p_scalar, w_vec, expectations_df, family_id_vector
)
print(f'Stimulation mismatch returned: {stim_mismatch_pct:.2f}%')

Input mechanical -> Node TGFB: effect on node increase, compared w/ data match, (correct so far=1, total=1), change=0.5767
Input mechanical -> Node B1int: effect on node increase, compared w/ data match, (correct so far=2, total=2), change=0.6970
Input mechanical -> Node Factin: effect on node increase, compared w/ data match, (correct so far=3, total=3), change=0.9933
Input mechanical -> Node migration: effect on node increase, compared w/ data match, (correct so far=4, total=4), change=0.1159
Input mechanical -> Node proliferation: effect on node increase, compared w/ data match, (correct so far=5, total=5), change=0.3961
Input mechanical -> Node aSMA: effect on node increase, compared w/ data match, (correct so far=6, total=6), change=0.9850
Input mechanical -> Node TIMP1: effect on node increase, compared w/ data match, (correct so far=7, total=7), change=0.3338
Input mechanical -> Node TIMP2: effect on node increase, compared w/ data match, (correct so far=8, total=8), change=0.33

In [6]:
# Inspect the two CSVs and check row counts.
def inspect_log(path, expected_data_rows, match_col):
    with open(path, newline='') as f:
        rows = list(csv.reader(f))
    n_total = len(rows)
    header = rows[0]
    data = rows[1:]
    n_data = len(data)
    n_match = sum(1 for r in data if r[match_col].strip() == 'match')
    n_notmatch = sum(1 for r in data if r[match_col].strip() == 'not match')
    return {
        'path': path,
        'total_rows': n_total,
        'header': header,
        'data_rows': n_data,
        'expected_data_rows': expected_data_rows,
        'matches': n_match,
        'not_matches': n_notmatch,
    }

ko_info   = inspect_log('KO_mismatch_log.csv', expected_data_rows=30, match_col=4)
stim_info = inspect_log('input_stimulation_mismatch_log.csv', expected_data_rows=62, match_col=3)

for info in (ko_info, stim_info):
    print(f"--- {info['path']} ---")
    print(f"  header:      {info['header']}")
    print(f"  total rows:  {info['total_rows']}  (expected {info['expected_data_rows']+1} = 1 header + {info['expected_data_rows']} data)")
    print(f"  data rows:   {info['data_rows']}  (expected {info['expected_data_rows']})")
    print(f"  matches:     {info['matches']}")
    print(f"  not matches: {info['not_matches']}")
    print()

--- KO_mismatch_log.csv ---
  header:      ['ko_condition', 'target_species', 'expected_behavior', 'model_behavior', 'label', 'delta']
  total rows:  31  (expected 31 = 1 header + 30 data)
  data rows:   30  (expected 30)
  matches:     28
  not matches: 2

--- input_stimulation_mismatch_log.csv ---
  header:      ['input_name', 'target_node', 'model_behavior', 'label', 'correct_so_far', 'total_so_far', 'delta']
  total rows:  63  (expected 63 = 1 header + 62 data)
  data rows:   62  (expected 62)
  matches:     47
  not matches: 15



In [7]:
# Assertions and PASS/FAIL summary.
EXPECTED_KO_ROWS    = 30
EXPECTED_STIM_ROWS  = 62
EXPECTED_KO_MATCH   = 28
EXPECTED_STIM_MATCH = 47

checks = []
checks.append(('KO log has exactly 30 data rows (no doubling)',
               ko_info['data_rows'] == EXPECTED_KO_ROWS))
checks.append(('Stim log has exactly 62 data rows (no doubling)',
               stim_info['data_rows'] == EXPECTED_STIM_ROWS))
checks.append((f'KO matches == {EXPECTED_KO_MATCH}/{EXPECTED_KO_ROWS}',
               ko_info['matches'] == EXPECTED_KO_MATCH))
checks.append((f'Stim matches == {EXPECTED_STIM_MATCH}/{EXPECTED_STIM_ROWS}',
               stim_info['matches'] == EXPECTED_STIM_MATCH))

print('Verification results')
print('=' * 60)
all_ok = True
for label, ok in checks:
    status = 'PASS' if ok else 'FAIL'
    if not ok:
        all_ok = False
    print(f'  [{status}] {label}')
print('=' * 60)
print('Combined:',
      f"{ko_info['matches'] + stim_info['matches']}/"
      f"{ko_info['data_rows'] + stim_info['data_rows']}",
      f"= {(ko_info['matches'] + stim_info['matches']) / (ko_info['data_rows'] + stim_info['data_rows']) * 100:.1f}%")
print()
print('OVERALL:', 'ALL CHECKS PASSED' if all_ok else 'SOME CHECKS FAILED')

Verification results
  [PASS] KO log has exactly 30 data rows (no doubling)
  [PASS] Stim log has exactly 62 data rows (no doubling)
  [PASS] KO matches == 28/30
  [PASS] Stim matches == 47/62
Combined: 75/92 = 81.5%

OVERALL: ALL CHECKS PASSED
